## Лабораторная 1: реализация шифра Цезаря, Вижинера и перестановочного шифра

In [ ]:
from src import CaesarCipher, VigenereCipher, PermutationCipher

### 1. Шифр цезаря

In [ ]:
caesar = CaesarCipher(alphabet="en")
text = "hello"

encrypted = caesar.encrypt(text, 3)
decrypted = caesar.decrypt(encrypted, 3)

print("text: hello, key: 3")
print("encrypted:", encrypted)
print("decrypted:", decrypted)
assert decrypted == text

In [ ]:
caesar = CaesarCipher(alphabet="ru")
text = "яблоко"

encrypted = caesar.encrypt(text, 3)
decrypted = caesar.decrypt(encrypted, 3)

print("text: яблоко, key: 3")
print("encrypted:", encrypted)
print("decrypted:", decrypted)
assert decrypted == text

In [ ]:
caesar = CaesarCipher(alphabet="en")
text = "race car!"

encrypted = caesar.encrypt(text, 7)
decrypted = caesar.decrypt(encrypted, 7)

print("text: race car!, key: 7")
print("encrypted:", encrypted)
print("decrypted:", decrypted)
assert decrypted == text

In [ ]:
caesar = CaesarCipher(alphabet="ru")
text = "хотите верити хотите не верити"
encrypted = caesar.encrypt(text, 32)
decrypted = caesar.decrypt(encrypted, 32)

print("text: хотите верити хотите не верити, key: 32")
print("encrypted:", encrypted)
print("decrypted:", decrypted)

assert decrypted == text

### 2. Шифр Виженера

In [ ]:
vigenere = VigenereCipher(alphabet="ru")
text = "криптографир"
encrypted = vigenere.encrypt(text, "шифр")
decrypted = vigenere.decrypt(encrypted, "шифр")
print("text: криптография, key: шифр")
print("encrypted:", encrypted)
print("decrypted:", decrypted)
assert encrypted == "гщэакччбшээп"
assert decrypted == text

In [ ]:
vigenere = VigenereCipher(alphabet="ru")
text = "шифр виженера"
encrypted = vigenere.encrypt(text, "лопух")
decrypted = vigenere.decrypt(encrypted, "лопух")
print("text: шифр виженера, key: лопух")
print("encrypted:", encrypted)
print("decrypted:", decrypted)
assert decrypted == text

In [ ]:
vigenere = VigenereCipher(alphabet="en")
text = "crypto-boss"
encrypted = vigenere.encrypt(text, "www")
decrypted = vigenere.decrypt(encrypted, "www")
print("text: crypto-boss, key: www")
print("encrypted:", encrypted)
print("decrypted:", decrypted)
assert decrypted == text

### 3. Перестановочный шифр

In [23]:
permutation = PermutationCipher()
key = (3, 1, 4, 2)
text = "crypto is good"
encrypted = permutation.encrypt(text, key)
decrypted = permutation.decrypt(encrypted, key)
print(encrypted)
print(decrypted)
assert encrypted == "ycpritsoogdo"
assert decrypted == text.replace(" ", "")

- блок: cryp
- блок: tois
- блок: good
ycpritsoogdo
cryptoisgood


In [ ]:
text = "ab x"
key = (3, 1, 4, 2)
original_length = len(text.replace(" ", ""))
encrypted = permutation.encrypt(text, key)
print('encrypted:', encrypted)
print("decrypted с паддингом:", permutation.decrypt(encrypted, key))
decrypted = permutation.decrypt(encrypted, key, original_length=original_length)
print("decrypted без паддинга:", decrypted)
assert decrypted == "abx"

In [ ]:
text = "book book hook"
key = (2, 1, 4, 3)
original_length = len(text.replace(" ", ""))
encrypted = permutation.encrypt(text, key)
print('encrypted:', encrypted)
print("decrypted с паддингом:", permutation.decrypt(encrypted, key))
decrypted = permutation.decrypt(encrypted, key, original_length=original_length)
print("decrypted без паддинга:", decrypted)

## Лабораторная 2: сеть Фейстеля и лавинный эффект

Блок — 16 бит, две половины по 8 бит, ключ — целое число 0..65535.
По умолчанию 8 раундов (минимум 4). Текст кодируется в UTF-8,
дополнение — PKCS#7, результат — hex-строка. Финальной перестановки нет.
Это учебный шифр, не предназначенный для защиты данных.

In [ ]:
from src import FeistelCipher

feistel = FeistelCipher(rounds=8)
feistel_key = 0xBEEF

### 1. Шифрование и расшифрование сообщений

Проверяем пустой текст, неполный и полный блок, несколько блоков,
русский текст, Unicode и настоящие байты заполнителя в конце сообщения.

In [ ]:
feistel_messages = ["", "A", "AB", "Hello, world!", "Привет, мир!", "🙂", "text\x01\x02"]
for message in feistel_messages:
    encrypted = feistel.encrypt(message, feistel_key)
    decrypted = feistel.decrypt(encrypted, feistel_key)
    print(f"{message!r} -> {encrypted} -> {decrypted!r}; D(E(m)) = m: {decrypted == message}")
    assert decrypted == message
    assert len(encrypted) % 4 == 0

### 2. Проверка одного блока и разных чисел раундов

Методы `encrypt_block` и `decrypt_block` работают с 16-битными числами
без UTF-8 и дополнения.

In [ ]:
for round_count in (4, 8, 12):
    block_cipher = FeistelCipher(rounds=round_count)
    for block in (0x0000, 0x1234, 0xFFFF):
        encrypted_block = block_cipher.encrypt_block(block, feistel_key)
        decrypted_block = block_cipher.decrypt_block(encrypted_block, feistel_key)
        print(f"rounds={round_count}: {block:04x} -> {encrypted_block:04x} -> {decrypted_block:04x}")
        assert decrypted_block == block

### 3. Проверка ошибок входных данных

In [ ]:
invalid_cases = [
    ("слишком мало раундов", lambda: FeistelCipher(rounds=3)),
    ("ключ вне диапазона", lambda: feistel.encrypt("test", 65536)),
    ("ключ не целое число", lambda: feistel.encrypt("test", "secret")),
    ("некорректный hex", lambda: feistel.decrypt("zzzz", feistel_key)),
    ("неполный шифроблок", lambda: feistel.decrypt("ab", feistel_key)),
    ("неверное дополнение", lambda: feistel.decrypt(
        f"{feistel.encrypt_block(0x4100, feistel_key):04x}", feistel_key)),
]
for description, operation in invalid_cases:
    try:
        operation()
    except ValueError as error:
        print(f"{description}: {error}")
    else:
        raise AssertionError(f"Ожидалась ошибка: {description}")

### 4. Лавинный эффект

Фиксируем блок `0x1234` и ключ `0xBEEF`. Последовательно меняем
каждый из 16 бит блока, затем каждый из 16 бит ключа. Бит 0 — младший.
Расстояние Хэмминга равно числу единиц в XOR шифротекстов.
Эксперимент выполняется на одном блоке без дополнения.

In [ ]:
avalanche_rows = feistel.avalanche_experiment(0x1234, feistel_key)
print("Что    Бит  Блок  Ключ  C     C_new Отличий")
for row in avalanche_rows:
    print(f"{row['target']:5}  {row['bit']:2}   {row['changed_block']:04x}  "
          f"{row['changed_key']:04x}  {row['ciphertext']:04x}  "
          f"{row['changed_ciphertext']:04x}  {row['different_bits']:2}/16")
    assert (row["block"] ^ row["changed_block"]).bit_count() + (row["key"] ^ row["changed_key"]).bit_count() == 1
    assert row["different_bits"] == (row["ciphertext"] ^ row["changed_ciphertext"]).bit_count()

for target in ("block", "key"):
    counts = [row["different_bits"] for row in avalanche_rows if row["target"] == target]
    average = sum(counts) / len(counts)
    print(f"Среднее ({target}): {average:.2f}/16 ({average / 16:.2%}); min={min(counts)}, max={max(counts)}")

### 5. Влияние числа раундов

Средние значения для одного блока и ключа не доказывают криптостойкость.
Учебная конструкция не обязана давать ровно 50% изменённых битов.

In [ ]:
for round_count in (4, 8, 12):
    rows = FeistelCipher(rounds=round_count).avalanche_experiment(0x1234, feistel_key)
    means = []
    for target in ("block", "key"):
        counts = [row["different_bits"] for row in rows if row["target"] == target]
        means.append(sum(counts) / len(counts))
    print(f"Раундов: {round_count}; изменение блока: {means[0]:.2f}/16; ключа: {means[1]:.2f}/16")